In [1]:
import recombigraph as rg
import subprocess
import os
import numpy as np
import pandas as pd
import toyplot

In [2]:
popsize = 5
curr_gen = ["P"+str(i) for i in range(popsize)]
gen_list = [[i,"NA","NA"] for i in curr_gen]

gens = 1

for gen in range(1,gens+1):
    names = ["F"+str(gen)+"_"+ str(i) for i in range(popsize)]
    # randomly choose two parents from previous generation for each individual
    # otherwise, could pair up ahead of time, and then sample from those pairs (ie "mate for life")
    parents = [list(np.random.choice(curr_gen,2,replace=True)) for i in range(popsize)] 
    for indiv_idx in range(popsize):
        parent1,parent2 = parents[indiv_idx]
        name_indiv = names[indiv_idx]
        gen_list.append([name_indiv,parent1,parent2])
    curr_gen = names
    
pedfile_text = 'Name Parent1 Parent2\n' + '\n'.join([' '.join(line) for line in gen_list])

# show our random pedigree -- this is what we will hand to pedigreeSim
print(pedfile_text)

Name Parent1 Parent2
P0 NA NA
P1 NA NA
P2 NA NA
P3 NA NA
P4 NA NA
F1_0 P0 P2
F1_1 P1 P2
F1_2 P0 P0
F1_3 P0 P1
F1_4 P2 P0


# PedigreeSim

In [3]:
test_dir = './PedigreeSim_results'

# make output directory if it doesn't exist
os.makedirs(test_dir, exist_ok=True)

ploidy = 2
mapfunction = "HALDANE"
pedfile = "example.ped"
chromfile = "example.chrom"
outputprefix = "example_out"

params = '''PLOIDY = {ploidy}
MAPFUNCTION = {mapfunction} 
PEDFILE = {pedfile}
CHROMFILE = {chromfile}
OUTPUT = {outputprefix}'''.format(ploidy = ploidy, 
                      mapfunction = mapfunction,
                      pedfile = os.path.join(test_dir,pedfile),
                      chromfile = os.path.join(test_dir,chromfile),
                      outputprefix = os.path.join(test_dir,outputprefix),
                           )

with open(os.path.join(test_dir,'example.params'),'w') as f:
    f.write(params)
    
# make the pedfile
with open(os.path.join(test_dir,'example.ped'), 'w') as f:
    f.write(pedfile_text)
    
# make chromosome file
chrom = '''
chromosome length centromere
A 100.0 50.0
'''
with open(os.path.join(test_dir,'example.chrom'), 'w') as f:
    f.write(chrom)
    
p = subprocess.Popen(["java", "-jar",os.path.join('./bin',"PedigreeSim.jar"),os.path.join(test_dir,'example.params')], stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)

output, errors = p.communicate()

print(output)

Parameter file=example.params
write file ./PedigreeSim_results/example_out.hsa
write file ./PedigreeSim_results/example_out.hsb
write file ./PedigreeSim_results/example_out_meioticconfigs.dat



In [4]:
# this records the founder haplotype for each segment
hsa = pd.read_table(os.path.join(test_dir,'example_out.hsa'),delimiter='\t',header=None)
hsa

,0,1,2,3,4,5,6
0,P0,A,1,0,NaN,NaN,NaN
1,P0,A,2,1,NaN,NaN,NaN
2,P1,A,1,2,NaN,NaN,NaN
3,P1,A,2,3,NaN,NaN,NaN
4,P2,A,1,4,NaN,NaN,NaN
5,P2,A,2,5,NaN,NaN,NaN
6,P3,A,1,6,NaN,NaN,NaN
7,P3,A,2,7,NaN,NaN,NaN
8,P4,A,1,8,NaN,NaN,NaN
9,P4,A,2,9,NaN,NaN,NaN


In [5]:
# this records the locations of segment boundaries
hsb = pd.read_table(os.path.join(test_dir,'example_out.hsb'),delimiter='\t',header=None)
hsb

,0,1,2,3
0,NaN,NaN,NaN,NaN
1,NaN,NaN,NaN,NaN
2,NaN,NaN,NaN,NaN
3,NaN,NaN,NaN,NaN
4,NaN,NaN,NaN,NaN
5,NaN,NaN,NaN,NaN
6,NaN,NaN,NaN,NaN
7,NaN,NaN,NaN,NaN
8,NaN,NaN,NaN,NaN
9,NaN,NaN,NaN,NaN


# How many segments, and how long, in final generation?

In [6]:
final_gen = hsb.iloc[10:,1:]

In [7]:
# convert the breakpoints from PedigreeSim hsb output to segments
def breaks_to_segments(hsb_row):
    curr_row = np.array(hsb_row[1])
    curr_row = curr_row[~np.isnan(curr_row)].tolist()
    edges = [0.0] + curr_row + [100]
    return [(edges[i], edges[i + 1]) for i in range(len(edges) - 1)]

all_seg_lengths = []
for row in final_gen.iterrows():
    segments = breaks_to_segments(row)
    seg_lengths = [i[1]-i[0] for i in segments]
    all_seg_lengths.append(seg_lengths)

### Num segments per chromatid in final generation:

In [8]:
[len(i) for i in all_seg_lengths]

[3, 1, 1, 3, 3, 2, 4, 1, 3, 1]

In [9]:
np.mean([len(i) for i in all_seg_lengths])

2.2

# Now wrap this up into something repeatable:
Specifically looking at average number of segments per chromatid in the final generation

In [10]:
mean_num_segments_PedigreeSim = []
for i in range(50):
    p = subprocess.Popen(["java", "-jar",os.path.join('./bin',"PedigreeSim.jar"),os.path.join(test_dir,'example.params')], stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)

    output, errors = p.communicate()

    hsb = pd.read_table(os.path.join(test_dir,'example_out.hsb'),delimiter='\t',header=None)
    final_gen = hsb.iloc[10:,1:]

    all_seg_lengths = []
    for row in final_gen.iterrows():
        segments = breaks_to_segments(row)
        seg_lengths = [i[1]-i[0] for i in segments]
        all_seg_lengths.append(seg_lengths)
    # how many segments per chromatid?
    n_segments = [len(i) for i in all_seg_lengths]
    mean_num_segments_PedigreeSim.append(np.mean(n_segments))

In [11]:
np.mean(mean_num_segments_PedigreeSim)

1.992

# Now do the recombigraph version:

In [12]:
mean_num_segments_recombigraph = []
for i in range(10000):
    model = rg.PedigreeModel(
        pedigree=gen_list,
        chromosomes={"A": 100.0},
    )

    # run simulation
    result = model.simulate()

    all_seg_lengths = []
    for indiv in ['F1_0','F1_1','F1_2','F1_3','F1_4']:
        homolog0 = result.individuals[indiv].homologs_by_chromosome['A'][0]
        homolog1 = result.individuals[indiv].homologs_by_chromosome['A'][1]
        all_seg_lengths.append(len(homolog0.segments))
        all_seg_lengths.append(len(homolog1.segments))

    mean_num_segments_recombigraph.append(np.mean(all_seg_lengths))

In [13]:
np.mean(mean_num_segments_recombigraph)

1.9961099999999998

In [14]:
result.individuals

{'P0': SimIndividual(individual_id='P0', time=0, homologs_by_chromosome={'A': [Homolog(homolog_id=0, chromosome='A', individual_id='P0', time=0, length=100.0, segments=[Segment(left=0.0, right=100.0, parent_homolog_id=None, founder_homolog_id=0)]), Homolog(homolog_id=1, chromosome='A', individual_id='P0', time=0, length=100.0, segments=[Segment(left=0.0, right=100.0, parent_homolog_id=None, founder_homolog_id=1)])]}),
 'P1': SimIndividual(individual_id='P1', time=0, homologs_by_chromosome={'A': [Homolog(homolog_id=2, chromosome='A', individual_id='P1', time=0, length=100.0, segments=[Segment(left=0.0, right=100.0, parent_homolog_id=None, founder_homolog_id=2)]), Homolog(homolog_id=3, chromosome='A', individual_id='P1', time=0, length=100.0, segments=[Segment(left=0.0, right=100.0, parent_homolog_id=None, founder_homolog_id=3)])]}),
 'P2': SimIndividual(individual_id='P2', time=0, homologs_by_chromosome={'A': [Homolog(homolog_id=4, chromosome='A', individual_id='P2', time=0, length=100.

# Now re-do with deeper generation:

## PedigreeSim

In [15]:
popsize = 5
curr_gen = ["P"+str(i) for i in range(popsize)]
gen_list = [[i,"NA","NA"] for i in curr_gen]

gens = 3

for gen in range(1,gens+1):
    names = ["F"+str(gen)+"_"+ str(i) for i in range(popsize)]
    # randomly choose two parents from previous generation for each individual
    # otherwise, could pair up ahead of time, and then sample from those pairs (ie "mate for life")
    parents = [list(np.random.choice(curr_gen,2,replace=True)) for i in range(popsize)] 
    for indiv_idx in range(popsize):
        parent1,parent2 = parents[indiv_idx]
        name_indiv = names[indiv_idx]
        gen_list.append([name_indiv,parent1,parent2])
    curr_gen = names
    
pedfile_text = 'Name Parent1 Parent2\n' + '\n'.join([' '.join(line) for line in gen_list])

# show our random pedigree -- this is what we will hand to pedigreeSim
print(pedfile_text)

Name Parent1 Parent2
P0 NA NA
P1 NA NA
P2 NA NA
P3 NA NA
P4 NA NA
F1_0 P2 P4
F1_1 P3 P0
F1_2 P3 P1
F1_3 P2 P1
F1_4 P1 P3
F2_0 F1_0 F1_4
F2_1 F1_0 F1_0
F2_2 F1_2 F1_0
F2_3 F1_3 F1_3
F2_4 F1_3 F1_1
F3_0 F2_1 F2_0
F3_1 F2_1 F2_1
F3_2 F2_0 F2_1
F3_3 F2_3 F2_3
F3_4 F2_2 F2_2


In [16]:
test_dir = './PedigreeSim_results'

# make output directory if it doesn't exist
os.makedirs(test_dir, exist_ok=True)

ploidy = 2
mapfunction = "HALDANE"
pedfile = "example.ped"
chromfile = "example.chrom"
outputprefix = "example_out"

params = '''PLOIDY = {ploidy}
MAPFUNCTION = {mapfunction} 
PEDFILE = {pedfile}
CHROMFILE = {chromfile}
OUTPUT = {outputprefix}'''.format(ploidy = ploidy, 
                      mapfunction = mapfunction,
                      pedfile = os.path.join(test_dir,pedfile),
                      chromfile = os.path.join(test_dir,chromfile),
                      outputprefix = os.path.join(test_dir,outputprefix),
                           )

with open(os.path.join(test_dir,'example.params'),'w') as f:
    f.write(params)
    
# make the pedfile
with open(os.path.join(test_dir,'example.ped'), 'w') as f:
    f.write(pedfile_text)
    
# make chromosome file
chrom = '''
chromosome length centromere
A 100.0 50.0
'''
with open(os.path.join(test_dir,'example.chrom'), 'w') as f:
    f.write(chrom)

In [17]:
mean_num_segments_PedigreeSim = []
for i in range(50):
    p = subprocess.Popen(["java", "-jar",os.path.join('./bin',"PedigreeSim.jar"),os.path.join(test_dir,'example.params')], stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)

    output, errors = p.communicate()

    hsb = pd.read_table(os.path.join(test_dir,'example_out.hsb'),delimiter='\t',header=None)
    final_gen = hsb.iloc[30:,1:]

    all_seg_lengths = []
    for row in final_gen.iterrows():
        segments = breaks_to_segments(row)
        seg_lengths = [i[1]-i[0] for i in segments]
        all_seg_lengths.append(seg_lengths)
    # how many segments per chromatid?
    n_segments = [len(i) for i in all_seg_lengths]
    mean_num_segments_PedigreeSim.append(np.mean(n_segments))

In [18]:
np.mean(mean_num_segments_PedigreeSim)

3.9619999999999997

# Recombigraph:

In [19]:
mean_num_segments_recombigraph = []
for i in range(1000):
    model = rg.PedigreeModel(
        pedigree=gen_list,
        chromosomes={"A": 100.0},
    )

    # run simulation
    result = model.simulate()

    all_seg_lengths = []
    for indiv in ['F3_0','F3_1','F3_2','F3_3','F3_4']:
        homolog0 = result.individuals[indiv].homologs_by_chromosome['A'][0]
        homolog1 = result.individuals[indiv].homologs_by_chromosome['A'][1]
        all_seg_lengths.append(len(homolog0.segments))
        all_seg_lengths.append(len(homolog1.segments))

    mean_num_segments_recombigraph.append(np.mean(all_seg_lengths))

In [20]:
np.mean(mean_num_segments_recombigraph)

4.0035